In [1]:
from echo.tools.web_scraping import get_url_doc

seller = 'https://whatfix.com/'

seller_doc = get_url_doc([seller])
print(seller_doc[0].page_content)

:rocket:

Whatfix Named a Leader in The Forrester Wave™: Digital Adoption Platforms, Q4 2024 Read the full report now →

Whatfix

Whatfix

Products

Digital Adoption Platform

Create contextual in-app guidance in the flow of work, collect feedback and analyze usage.

Product Analytics

Analyze how users engage and use desktop and web applications with no-code event tracking.

Mirror

Create interactive sandbox environments of web applications for hands-on software training.

Solutions

By Use Case

Digital Transformation

Adopt new technology without a dip in productivity

Change Management

Manage enterprise change with efficiency

Employee Training

Train team members with in-app eLearning

User Adoption

Increase user adoption of your enterprise software

Employee Onboarding

Onboard new hires faster with in-app training

Performance Support

Improve employee productivity with self-service support

User Onboarding

Onboard new users faster with personalized walkthroughs

AI Adoption

In [2]:
from echo.data.utils import get_relevant_link_categories

categories = get_relevant_link_categories()

In [3]:
categories_str = "\n".join([f"{i+1}. {categories[cat]['Section']}\nPurpose: {categories[cat]['Purpose']}" for i, cat in enumerate(categories)])
print(categories_str)

1. Hero Headline & Subhead
Purpose: Core product promise and ICP target language
2. Features & Capabilities
Purpose: Aligns with buyer pains or tech stack; helps match capabilities to needs
3. Persona-Focused Pages
Purpose: Supports stakeholder mapping and personalized value framing
4. Use Case Pages
Purpose: Direct input to infer buyer initiatives and anchor business case sections
5. Customer Logos
Purpose: Provides social proof and lets us recommend relevant examples
6. Case Studies / Outcomes
Purpose: Fuels value prop messaging and ROI references
7. Blog & Thought Leadership
Purpose: Extracts tone, narrative, and company POV for outbound or strategic talk tracks
8. CTA / Conversion Copy
Purpose: Helps learn their sales messaging and urgency framing
9. Video & Webinar Content
Purpose: Supports buyer persona tone, narrative extraction, or sales talk tracks


In [4]:
from echo.tools.web_scraping import extract_links_from_url
from langchain_community.document_loaders import SeleniumURLLoader

links = extract_links_from_url(seller)
print("Loading links from total of ", len(links), " links")
link_docs = SeleniumURLLoader(urls=links).load()

Loading links from total of  63  links


In [7]:
from typing import List
from crewai import Agent, Task
from pydantic import BaseModel, Field
from echo.echo_agent import EchoAgent
from echo.utils import format_response, get_crew_llm


def get_nav_links(website_url: str, categories: str, links_str: str, num_links: int = 15) -> str:
    class Link(BaseModel):
        url: str = Field(..., title="URL of the navigation link")
        category: str = Field(
            ...,
            title="Category of the navigation link",
            description="The category of the navigation link",
        )
        rationale: str = Field(
            ...,
            title="Rationale for the category",
            description="The rationale for the category of the navigation link",
        )


    class CategorizedLinks(BaseModel):
        nav_links: List[Link] = Field(
            ...,
            title="Navigation links on the website from Link Object",
            description="List of navigation links on the website from Link Object",
        )



    agent = Agent(
            role="Website Crawling Expert",
            goal="Filter out the relevant links that might be relevant for an Account Executive to understand the company's offerings and services",
            backstory="You are an expert in extracting or filter out all the important links from a given website that might be relevant for an Account Executive to understand the company's offerings and services.",
            llm=get_crew_llm(),
        )

    task = Task(
        name="Categorizing Links",
        description=(
            "Given below the {website} landing page content and a list navigation links from {website}, extract out the most relevant links that might be relevant for an Account Executive to understand the company's offerings and services."
            "A link is relevant if it can belong to one of the categories below: \n"
            "{categories}"
            
            "\n---Website Content---\n"
            "{content}"
            "\n---End of Content---\n"
            
            "\n---Extracted Navigation Links---\n"
            "{links}"
            "\n---End of Navigation Links---\n"
            
            "Based on this information, filter only the relevant links that might be relevant for an Account Executive to understand the company's offerings and services.\n"
            "You need to extract the relevant link and also assign a category to the link.\n"
            "You also need to provide a rationale for the category assigned to the link.\n"
            "The category should be one and ONLY ONE of the categories provided\n"
            "The final output should have atmost {num_links} links."
            "The total links can be less than {num_links} IF A LOT OF LINKS ARE NOT RELEVANT but not more than that.\n"
        ),
        expected_output=(
            "The response should conform to the provided schema."
            "You need to extract the following information in the following pydantic structure -\n"
            "{pydantic_structure}\n"
        ),
        output_pydantic=CategorizedLinks,
        agent=agent,
    )

    crew = EchoAgent(
        agents=[agent], 
        tasks=[task]
    )

    inputs = {
        "website": website_url,
        "content": seller_doc[0].page_content,
        "categories": categories,
        "links": links_str,
        "num_links": num_links,
    }

    response = crew.kickoff(inputs=inputs)
    response = format_response(response)
    return response


In [26]:
per_batch_num_links = 25
per_batch_max_filter_links = 10
max_webpage_links = 20

In [9]:
from langchain_core.documents import Document
from tqdm.auto import tqdm

def sanitize_url(url: str):
    return url.split('?')[0] if '?' in url else url

url_to_doc_map = {
    sanitize_url(link_doc.metadata["source"]): link_doc
    for link_doc in link_docs
}

In [10]:
print("\n".join(url_to_doc_map.keys()))

https://whatfix.com/solutions/elearning/
https://whatfix.com/pricing/
https://whatfix.com/resources/webinars/
https://whatfix.com/resources/videos/
https://whatfix.com/resources/analyst-reports/forrester-wave-digital-adoption-platforms-q4-2024/
https://whatfix.com/solutions/digital-transformation/
https://whatfix.com/sign-in/
https://whatfix.com/solutions/s2p/
https://whatfix.com/solutions/performance-support/
https://whatfix.com/solutions/sales-professionals/
https://whatfix.com/innovation/userization/
https://whatfix.com/awards/
https://whatfix.com/products/
https://whatfix.com/solutions/digital-banking/
https://whatfix.com/request-trial/
https://whatfix.com/solutions/ai-adoption/
https://whatfix.com/support/faq/
https://whatfix.com/solutions/user-onboarding/
https://whatfix.com/solutions/self-service-support/
https://whatfix.com/resources/analyst-reports/
https://whatfix.com/blog/
https://whatfix.com/trust/
https://whatfix.com/customers/
https://whatfix.com/partners/
https://whatfix

In [ ]:
all_links: List[Document] = list()
for i in tqdm(range(0, len(link_docs), per_batch_num_links), desc="Extracting Navigation Links"):
    
    links_str = "\n\n".join([(
            f"URL: {sanitize_url(link_doc.metadata['source'])}\n"
            f"Title: {link_doc.metadata['title']}\n"
            f"Description: {link_doc.metadata['description']}"
        )
        for link_doc in link_docs[i:i + per_batch_num_links]
    ])
    navbar_links = get_nav_links(
        website_url=seller,
        categories=categories_str,
        links_str=links_str,
        num_links=per_batch_max_filter_links,
    )
    url_docs = [link["url"] for link in navbar_links["nav_links"]]
    print("Total links in this batch: ", len(url_docs))
    all_links.extend(url_docs)

all_links = list(set(all_links))

Extracting Navigation Links:   0%|          | 0/3 [00:00<?, ?it/s]

Overriding of current TracerProvider is not allowed
Overriding of current TracerProvider is not allowed


Total links in this batch:  10


Overriding of current TracerProvider is not allowed


Total links in this batch:  10
Total links in this batch:  9


In [20]:
len(all_links), len(set(all_links))

(29, 21)

In [21]:
all_links_docs = SeleniumURLLoader(urls=all_links).load()

In [ ]:
all_links = all_links[:100]
links_str = "\n\n".join([(
        f"URL: {link_doc.metadata['source']}\n"
        f"Title: {link_doc.metadata['title']}\n"
        f"Description: {link_doc.metadata['description']}"
    )
    for link_doc in all_links_docs
])

print("Getting final links")
navbar_links = get_nav_links(
    website_url=seller,
    categories=categories_str,
    links_str=links_str,
    num_links=max_webpage_links,
)
final_links = [
    (link["url"], link["category"], link["rationale"]) 
    for link in navbar_links["nav_links"]
]

print("Total Final Links:", len(final_links))
final_links_docs = SeleniumURLLoader(urls=[link[0] for link in final_links]).load()
for final_link_doc, final_link in zip(final_links_docs, final_links):
    final_link_doc.metadata["category"] = final_link[1]
    final_link_doc.metadata["rationale"] = final_link[2]

Overriding of current TracerProvider is not allowed


Getting final links
Final Links: [('https://whatfix.com/products/', 'Features & Capabilities'), ('https://whatfix.com/solutions/', 'Use Case Pages'), ('https://whatfix.com/customers/', 'Customer Logos'), ('https://whatfix.com/resources/case-studies/', 'Case Studies / Outcomes'), ('https://whatfix.com/blog/', 'Blog & Thought Leadership'), ('https://whatfix.com/products/digital-adoption-platform/', 'Features & Capabilities'), ('https://whatfix.com/products/product-analytics/', 'Features & Capabilities'), ('https://whatfix.com/solutions/digital-transformation/', 'Use Case Pages'), ('https://whatfix.com/solutions/user-onboarding/', 'Use Case Pages'), ('https://whatfix.com/pricing/', 'CTA / Conversion Copy'), ('https://whatfix.com/request-demo/', 'CTA / Conversion Copy')]


In [ ]:
from echo.data.utils import get_category_prompts

category_prompts = get_category_prompts()
for final_link_doc in final_links_docs:
    final_link_doc.metadata["category_prompt"] = category_prompts[final_link_doc.metadata["category"]]

In [39]:
import concurrent.futures

from echo.settings import MAX_CONCURRENT_REQUESTS
from echo.tools.web_scraping import extract_data_from_webpage


DATA_EXTRACTION_SYS_PROMPT = """
You are an expert in sales such that you can extract out the content from a website of a company that would be relevant to a potential client of that company. 
"""
PER_CATEGORY_DATA_EXTRACTION_USER_PROMPT = """
Given the website content, extract out the information according to the instruction provided below.

---Website Content---
{content}
---End of Content---

---Instruction---
{instruction}
---End of Instruction---
"""

def extract_per_category_data_from_links(
    links: List[Document],
    system_prompt: str = DATA_EXTRACTION_SYS_PROMPT,
):
    extracted_results = []

    def process_link(link: Document):
        print("Processing link:", link)
        try:
            # Construct the full URL if necessary.
            # Extract the text from the URL (assumes extract_text_from_url is defined elsewhere)
            extracted_text = (
                f"URL: {link.metadata['source']}\n"
                f"Title: {link.metadata['title']}\n"
                f"Description: {link.metadata['description']}\n"
                f"Page Content: {link.metadata['category']}\n"
            )
            if not extracted_text:
                print(f"Failed to extract text from {link}")
                return None
            # Use the extracted text to get client-focused data
            print("Extracting data from:", link)
            print("Extracted text:", extracted_text)
            extraction_prompt = PER_CATEGORY_DATA_EXTRACTION_USER_PROMPT.format(
                content=extracted_text,
                instruction=link.metadata["category_prompt"]
            )
            extracted_data = extract_data_from_webpage(
                extraction_prompt, 
                system_prompt=system_prompt
            )
            print(f"Extracted data from {link}")
            return {"link": link, "data": extracted_data}
        except Exception as e:
            print(f"Error processing link {link}: {e}")
            return None

    # Use ThreadPoolExecutor to process links concurrently.
    with concurrent.futures.ThreadPoolExecutor(
        max_workers=MAX_CONCURRENT_REQUESTS
    ) as executor:
        # Submit all tasks and store futures in a dictionary.
        futures = {executor.submit(process_link, link): link for link in links}
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=len(futures),
            desc="Summarizing Data From Links",
        ):
            result = future.result()
            if result is not None:
                extracted_results.append(result)

    # Filter out None results
    extracted_results = [result for result in extracted_results if result is not None]
    return extracted_results


In [ ]:
extracted_results = extract_per_category_data_from_links(final_links_docs)

In [46]:
for r in extracted_results:
    print(r["link"].metadata["source"])
    print(r["link"].metadata["category_prompt"])
    print(r["data"])
    print("="*80)

https://whatfix.com/resources/case-studies/
Type of content: Case Studies / Outcomes
Task: For each case study, extract the company name, persona quoted, challenge, solution, and quantifiable outcome. Provide summary in 3 bullet points per study.
Purpose: Fuels value prop messaging and ROI references
Examples: Quotes, challenges, results
(e.g. “+40% demo conversion”)

I'm unable to extract specific case study details such as company name, persona quoted, challenge, solution, and quantifiable outcomes from the provided content because the detailed case studies or outcomes are not included in the text. The content merely describes the purpose of the webpage (Customer Success Stories) and offers a brief general description of what Whatfix aims to achieve with its case studies.

To fulfill your request accurately, I would need detailed information from the actual case studies listed on that webpage. If you have access to specific case studies from the Whatfix website, you could share those